# 🎯 Train YOLOv8n Face Detection trên WIDER Face Dataset

Notebook này thực hiện:
1. Cài đặt thư viện cần thiết (ultralytics)
2. Tải model YOLOv8n pretrained
3. Tải và chuẩn bị dataset WIDER Face → format YOLO
4. Huấn luyện model
5. Đánh giá và xuất model

> ⚠️ Chạy trên Google Colab với GPU (T4 trở lên).

## 1. Cài đặt thư viện

In [ ]:
!pip install -q ultralytics gdown

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Tải model YOLOv8n pretrained

In [ ]:
from ultralytics import YOLO

# Tải model YOLOv8n pretrained trên COCO
model = YOLO('yolov8n.pt')
print("✅ Đã tải model YOLOv8n pretrained.")
print(model.info())

## 3. Tải dataset WIDER Face

WIDER Face gồm 3 phần:
- **Train**: ~12,880 ảnh, ~159k faces
- **Val**: ~3,226 ảnh
- **Annotations**: bounding boxes

In [ ]:
import os

DATASET_DIR = '/content/wider_face'
os.makedirs(DATASET_DIR, exist_ok=True)

# --- Tải WIDER Face Train ---
if not os.path.exists(f'{DATASET_DIR}/WIDER_train'):
    print("📥 Đang tải WIDER Face Train...")
    !gdown --id 15hGDLhsx8bLgLcIRD5DhYt5iBxnjNF1M -O {DATASET_DIR}/WIDER_train.zip
    !unzip -q {DATASET_DIR}/WIDER_train.zip -d {DATASET_DIR}
    !rm {DATASET_DIR}/WIDER_train.zip
    print("✅ WIDER Face Train đã tải xong.")
else:
    print("✅ WIDER Face Train đã tồn tại.")

# --- Tải WIDER Face Val ---
if not os.path.exists(f'{DATASET_DIR}/WIDER_val'):
    print("📥 Đang tải WIDER Face Val...")
    !gdown --id 1GUCogbp16PMGa39thoMMeWxp7Rp5oM8Q -O {DATASET_DIR}/WIDER_val.zip
    !unzip -q {DATASET_DIR}/WIDER_val.zip -d {DATASET_DIR}
    !rm {DATASET_DIR}/WIDER_val.zip
    print("✅ WIDER Face Val đã tải xong.")
else:
    print("✅ WIDER Face Val đã tồn tại.")

# --- Tải Annotations ---
if not os.path.exists(f'{DATASET_DIR}/wider_face_split'):
    print("📥 Đang tải Annotations...")
    !gdown --id 1jdZgruKQm87ICRhROolDix2koME5JVAj -O {DATASET_DIR}/wider_face_split.zip
    !unzip -q {DATASET_DIR}/wider_face_split.zip -d {DATASET_DIR}
    !rm {DATASET_DIR}/wider_face_split.zip
    print("✅ Annotations đã tải xong.")
else:
    print("✅ Annotations đã tồn tại.")

print("\n📂 Cấu trúc dataset:")
!ls -la {DATASET_DIR}

## 4. Chuyển đổi WIDER Face → YOLO format

WIDER Face annotation format:
```
file_name
number_of_faces
x1 y1 w h blur expression illumination invalid occlusion pose
```

YOLO format: `class x_center y_center width height` (normalized 0-1)

In [ ]:
import os
from pathlib import Path
from PIL import Image

def convert_wider_to_yolo(annotation_path, image_root, output_image_dir, output_label_dir, min_face_size=10):
    """
    Chuyển đổi WIDER Face annotation sang YOLO format.
    
    Args:
        annotation_path: đường dẫn file annotation (wider_face_train_bbx_gt.txt)
        image_root: thư mục gốc chứa ảnh (WIDER_train/images hoặc WIDER_val/images)
        output_image_dir: thư mục output cho ảnh
        output_label_dir: thư mục output cho labels
        min_face_size: kích thước tối thiểu của face (bỏ qua face quá nhỏ)
    
    Returns:
        dict: thống kê {total_images, total_faces, skipped_faces}
    """
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)
    
    stats = {'total_images': 0, 'total_faces': 0, 'skipped_faces': 0}
    
    with open(annotation_path, 'r') as f:
        while True:
            # Đọc tên file ảnh
            line = f.readline().strip()
            if not line:
                break
            
            image_rel_path = line
            image_path = os.path.join(image_root, image_rel_path)
            
            # Đọc số lượng faces
            num_faces = int(f.readline().strip())
            
            # Nếu 0 faces, WIDER vẫn có 1 dòng dummy → đọc bỏ
            if num_faces == 0:
                f.readline()  # đọc dòng dummy
                continue
            
            # Kiểm tra file ảnh tồn tại
            if not os.path.exists(image_path):
                # Bỏ qua các dòng annotation
                for _ in range(num_faces):
                    f.readline()
                continue
            
            # Lấy kích thước ảnh
            try:
                img = Image.open(image_path)
                img_w, img_h = img.size
                img.close()
            except Exception as e:
                print(f"⚠️ Không mở được {image_path}: {e}")
                for _ in range(num_faces):
                    f.readline()
                continue
            
            yolo_labels = []
            
            for _ in range(num_faces):
                bbox_info = f.readline().strip().split()
                x1 = int(bbox_info[0])
                y1 = int(bbox_info[1])
                w = int(bbox_info[2])
                h = int(bbox_info[3])
                
                # Bỏ qua invalid faces (cột 7) và faces quá nhỏ
                invalid = int(bbox_info[7]) if len(bbox_info) > 7 else 0
                if invalid == 1:
                    stats['skipped_faces'] += 1
                    continue
                
                if w < min_face_size or h < min_face_size:
                    stats['skipped_faces'] += 1
                    continue
                
                # Clamp bbox vào giới hạn ảnh
                x1 = max(0, x1)
                y1 = max(0, y1)
                w = min(w, img_w - x1)
                h = min(h, img_h - y1)
                
                if w <= 0 or h <= 0:
                    stats['skipped_faces'] += 1
                    continue
                
                # Chuyển sang YOLO format: class x_center y_center width height (normalized)
                x_center = (x1 + w / 2) / img_w
                y_center = (y1 + h / 2) / img_h
                w_norm = w / img_w
                h_norm = h / img_h
                
                # Clamp to [0, 1]
                x_center = min(max(x_center, 0), 1)
                y_center = min(max(y_center, 0), 1)
                w_norm = min(max(w_norm, 0), 1)
                h_norm = min(max(h_norm, 0), 1)
                
                # class 0 = face
                yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
                stats['total_faces'] += 1
            
            if yolo_labels:
                # Tạo tên file phẳng (thay / bằng _)
                flat_name = image_rel_path.replace('/', '_').replace('\\', '_')
                base_name = os.path.splitext(flat_name)[0]
                
                # Symlink hoặc copy ảnh
                dst_image = os.path.join(output_image_dir, flat_name)
                if not os.path.exists(dst_image):
                    os.symlink(os.path.abspath(image_path), dst_image)
                
                # Ghi label file
                label_path = os.path.join(output_label_dir, base_name + '.txt')
                with open(label_path, 'w') as lf:
                    lf.write('\n'.join(yolo_labels) + '\n')
                
                stats['total_images'] += 1
    
    return stats

print("✅ Hàm convert_wider_to_yolo đã sẵn sàng.")

In [ ]:
# ============================
# Chuyển đổi Train set
# ============================
YOLO_DIR = '/content/wider_yolo'

print("🔄 Đang chuyển đổi Train set...")
train_stats = convert_wider_to_yolo(
    annotation_path=f'{DATASET_DIR}/wider_face_split/wider_face_train_bbx_gt.txt',
    image_root=f'{DATASET_DIR}/WIDER_train/images',
    output_image_dir=f'{YOLO_DIR}/images/train',
    output_label_dir=f'{YOLO_DIR}/labels/train',
    min_face_size=10
)
print(f"✅ Train: {train_stats['total_images']} ảnh, {train_stats['total_faces']} faces, {train_stats['skipped_faces']} skipped")

# ============================
# Chuyển đổi Val set
# ============================
print("\n🔄 Đang chuyển đổi Val set...")
val_stats = convert_wider_to_yolo(
    annotation_path=f'{DATASET_DIR}/wider_face_split/wider_face_val_bbx_gt.txt',
    image_root=f'{DATASET_DIR}/WIDER_val/images',
    output_image_dir=f'{YOLO_DIR}/images/val',
    output_label_dir=f'{YOLO_DIR}/labels/val',
    min_face_size=10
)
print(f"✅ Val: {val_stats['total_images']} ảnh, {val_stats['total_faces']} faces, {val_stats['skipped_faces']} skipped")

## 5. Kiểm tra dữ liệu chuyển đổi

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import numpy as np
from PIL import Image

def visualize_yolo_sample(image_dir, label_dir, n_samples=4):
    """Hiển thị một số mẫu để kiểm tra chất lượng chuyển đổi."""
    images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]
    samples = random.sample(images, min(n_samples, len(images)))
    
    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
    if len(samples) == 1:
        axes = [axes]
    
    for ax, img_name in zip(axes, samples):
        img = Image.open(os.path.join(image_dir, img_name))
        img_w, img_h = img.size
        img_np = np.array(img)
        
        ax.imshow(img_np)
        
        # Đọc labels
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(label_dir, label_name)
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, xc, yc, w, h = map(float, parts)
                        # Convert YOLO → pixel coords
                        x1 = (xc - w/2) * img_w
                        y1 = (yc - h/2) * img_h
                        bw = w * img_w
                        bh = h * img_h
                        rect = patches.Rectangle((x1, y1), bw, bh,
                                                 linewidth=2, edgecolor='lime', facecolor='none')
                        ax.add_patch(rect)
        
        ax.set_title(img_name[:30], fontsize=8)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_yolo_sample(f'{YOLO_DIR}/images/train', f'{YOLO_DIR}/labels/train', n_samples=4)

## 6. Tạo file cấu hình dataset YAML

In [ ]:
import yaml

dataset_yaml = {
    'path': YOLO_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['face']
}

yaml_path = f'{YOLO_DIR}/wider_face.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"✅ Dataset YAML đã tạo: {yaml_path}")
print("---")
with open(yaml_path, 'r') as f:
    print(f.read())

## 7. Huấn luyện YOLOv8n Face Detection

| Param | Giá trị | Ghi chú |
|-------|---------|--------|
| `epochs` | 100 | Có thể tăng nếu cần |
| `imgsz` | 640 | Standard YOLO size |
| `batch` | 16 | Giảm nếu OOM trên T4 |
| `lr0` | 0.01 | Learning rate ban đầu |
| `patience` | 20 | Early stopping |

In [ ]:
from ultralytics import YOLO

# ============================
# Cấu hình Training
# ============================
EPOCHS = 100          # Số epoch
BATCH_SIZE = 16       # Giảm xuống 8 nếu OOM
IMG_SIZE = 640        # Kích thước ảnh đầu vào
LR0 = 0.01           # Learning rate ban đầu
LRF = 0.001          # Learning rate cuối
PATIENCE = 20         # Early stopping patience
WORKERS = 4           # DataLoader workers

# Tải model base
model = YOLO('yolov8n.pt')

# Huấn luyện
results = model.train(
    data=f'{YOLO_DIR}/wider_face.yaml',
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    lr0=LR0,
    lrf=LRF,
    patience=PATIENCE,
    workers=WORKERS,
    device=0,
    project='/content/yolov8_face',
    name='wider_face_v1',
    exist_ok=True,
    # Augmentation
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    # Logging
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

## 8. Đánh giá model

In [ ]:
# Đánh giá trên val set
best_model = YOLO('/content/yolov8_face/wider_face_v1/weights/best.pt')
metrics = best_model.val(
    data=f'{YOLO_DIR}/wider_face.yaml',
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)

print("\n📊 Kết quả đánh giá:")
print(f"  mAP@50:     {metrics.box.map50:.4f}")
print(f"  mAP@50-95:  {metrics.box.map:.4f}")
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")

In [ ]:
# Hiển thị training curves
from IPython.display import Image as IPImage, display
import os

result_dir = '/content/yolov8_face/wider_face_v1'

plots = ['results.png', 'confusion_matrix.png', 'val_batch0_pred.png', 'P_curve.png', 'R_curve.png']
for plot in plots:
    path = os.path.join(result_dir, plot)
    if os.path.exists(path):
        print(f"\n📈 {plot}")
        display(IPImage(filename=path, width=800))

## 9. Test inference trên ảnh mẫu

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import random

# Load best model
best_model = YOLO('/content/yolov8_face/wider_face_v1/weights/best.pt')

# Chọn ngẫu nhiên vài ảnh từ val set
val_images = [os.path.join(f'{YOLO_DIR}/images/val', f) 
              for f in os.listdir(f'{YOLO_DIR}/images/val') 
              if f.endswith(('.jpg', '.png'))]
test_images = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, img_path in zip(axes, test_images):
    results = best_model.predict(img_path, conf=0.25, iou=0.45, verbose=False)
    
    # Lấy ảnh với bbox đã vẽ
    plotted = results[0].plot()
    ax.imshow(plotted[:, :, ::-1])  # BGR → RGB
    
    n_faces = len(results[0].boxes)
    ax.set_title(f'{n_faces} faces detected', fontsize=10)
    ax.axis('off')

plt.suptitle('YOLOv8n Face Detection — Inference Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Xuất model & Lưu về Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục output
SAVE_DIR = '/content/drive/MyDrive/yolov8_face_models'
os.makedirs(SAVE_DIR, exist_ok=True)

import shutil

# Copy best weights
src_best = '/content/yolov8_face/wider_face_v1/weights/best.pt'
src_last = '/content/yolov8_face/wider_face_v1/weights/last.pt'

if os.path.exists(src_best):
    shutil.copy2(src_best, f'{SAVE_DIR}/yolov8n_face_best.pt')
    print(f"✅ Best model → {SAVE_DIR}/yolov8n_face_best.pt")

if os.path.exists(src_last):
    shutil.copy2(src_last, f'{SAVE_DIR}/yolov8n_face_last.pt')
    print(f"✅ Last model → {SAVE_DIR}/yolov8n_face_last.pt")

# Copy kết quả training
results_src = '/content/yolov8_face/wider_face_v1/results.csv'
if os.path.exists(results_src):
    shutil.copy2(results_src, f'{SAVE_DIR}/results.csv')
    print(f"✅ Results CSV → {SAVE_DIR}/results.csv")

print(f"\n📁 Tất cả đã lưu tại: {SAVE_DIR}")
!ls -la {SAVE_DIR}

## 11. (Tuỳ chọn) Export sang ONNX

Export ONNX để deploy trên các platform khác.

In [ ]:
# Export ONNX
best_model = YOLO('/content/yolov8_face/wider_face_v1/weights/best.pt')

onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
    opset=12,
)

print(f"\n✅ ONNX model exported: {onnx_path}")

# Copy sang Drive
if onnx_path and os.path.exists(onnx_path):
    shutil.copy2(onnx_path, f'{SAVE_DIR}/yolov8n_face_best.onnx')
    print(f"✅ ONNX → {SAVE_DIR}/yolov8n_face_best.onnx")

---

## 📝 Tóm tắt

| Bước | Mô tả |
|------|--------|
| 1 | Cài ultralytics |
| 2 | Tải YOLOv8n pretrained |
| 3 | Tải WIDER Face (Train + Val + Annotations) |
| 4 | Chuyển đổi sang YOLO format |
| 5 | Kiểm tra trực quan dữ liệu |
| 6 | Tạo dataset YAML |
| 7 | Train 100 epochs |
| 8 | Đánh giá mAP |
| 9 | Test inference |
| 10 | Lưu model về Drive |
| 11 | Export ONNX (tuỳ chọn) |

### Tips:
- Nếu bị **OOM** trên T4: giảm `BATCH_SIZE` xuống 8
- Muốn train lâu hơn: tăng `EPOCHS` lên 200
- Muốn chất lượng tốt hơn: dùng `yolov8s.pt` thay vì `yolov8n.pt`